In [1]:
from moviepy import VideoFileClip
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
import os
import google.generativeai as genai

C:\Users\Dell\AppData\Local\Temp\ipykernel_14060\3495285806.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
def extract_audio(video_path,output_audio_path):
    video = VideoFileClip(video_path)
    audio = video.audio
    audio.write_audiofile(output_audio_path)
    video.close()
    return output_audio_path

In [3]:
def audio2text(output_audio_path):
    model = WhisperModel("base",device="cpu",compute_type="int8")
    segments,info = model.transcribe(output_audio_path)
    segments = list(segments)
    return segments

In [4]:
def create_chunks(segments,chunk_size=4):
    chunks=[]
    for i in range(0,len(segments),chunk_size):
        text = ""
        for segment in segments[i:i+chunk_size]:
            text +=segment.text
        chunks.append({
            "text" : text.strip(),
            "start" : segments[i:i+chunk_size][0].start,
            "end" : segments[i:i+chunk_size][-1].end
            
        })
    return chunks

In [5]:
embeder = SentenceTransformer("all-MiniLM-L6-v2")
def embed_chunks(chunks):
    chunks_text = [chunk["text"] for chunk in chunks]
    embeddings = embeder.encode(chunks_text)
    return embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
def store_embeddings(embeddings,chunks):
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_or_create_collection(name="my_collection")
    collection.add(
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(chunks))],
        documents=[chunk["text"] for chunk in chunks],
        metadatas=[{
            "start" : chunk["start"],
            "end" : chunk["end"]
        } for chunk in chunks]
    )
    return collection

In [7]:
def ingestion_pipeline(video_path,output_audio_path):
    audio_path = extract_audio(video_path,output_audio_path)
    segments = audio2text(audio_path)
    chunks = create_chunks(segments)
    embedding = embed_chunks(chunks)
    collection = store_embeddings(embedding,chunks)
    return collection,chunks

In [10]:
collection,chunks = ingestion_pipeline("sample_video.mp4","output_sample_audio.wav")

MoviePy - Writing audio in output_sample_audio.wav


MoviePy - Done.


In [11]:
def embed_query(query):
    query_embedding= embeder.encode(query)
    return query_embedding

In [12]:
def retrive_chunks(collection,query_embedding):
    result = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results = 3
    )
    return result

In [13]:
def retrive_full_transcript(chunks):
    full_transcript =""
    for chunk in chunks:
        full_transcript+= chunk["text"]+"\n\n"
    return full_transcript

In [14]:
def generate_chunk_prompt(retrived_chunks,query):
    context = "\n\n".join(
        retrived_chunks["documents"][0]
    )
    prompt = f"""
    You are an AI Meeting Assistant.
    Your task is to answer the user's question using ONLY the provided meeting transcript context.
    Guidelines:
        - Answer only from the provided context.
        - Do not make up information.
        - If the answer is partially available, answer using only the available information.
        - If the answer is not found in the context, reply:
          "This information was not discussed in the meeting."
        - Keep the answer clear, concise, and professional.
        - If appropriate, mention the approximate timestamps available in the context.

    Meeting Transcript Context:
    {context}

    User Question:
    {query}

    Answer:
    """
    return prompt

In [15]:
def generate_transcript_prompt(full_transcript):
    prompt = f"""
    You are an AI Meeting Assistant.
    Your task is to analyze the complete meeting transcript and generate a structured summary.
    Include the following sections:

    1. Overall Summary
   - Briefly describe the purpose of the meeting.
    2. Key Discussion Points
   - List the major topics discussed.
    3. Decisions Made
   - Mention all important decisions taken during the meeting.
   - If none, write "No explicit decisions were made."
    4. Action Items
   - List any tasks assigned or follow-up work.
   - If none, write "No action items were identified."
    5. Deadlines
   - Mention any deadlines or important dates discussed.
   - If none, write "No deadlines were discussed."

    Keep the summary concise, well-structured, and easy to read.
    Do not invent information that is not present in the transcript.

    Meeting Transcript:

    {full_transcript}

    Summary:
    """

    return prompt

In [16]:
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash-lite")

In [17]:
def full_transcript_pipeline(chunks):
    full_transcript = retrive_full_transcript(chunks)
    prompt = generate_transcript_prompt(full_transcript)
    response = model.generate_content(prompt)
    return response.text

In [18]:
def normal_chunk_pipeline(query,collection):
    embedding = embed_query(query)
    retrived_chunks = retrive_chunks(collection,embedding)
    prompt = generate_chunk_prompt(retrived_chunks,query)
    response = model.generate_content(prompt)
    return response.text

In [19]:
def backend_pipeline(query,collection,chunks):
    keywords = [
        "summary",
        "summarize",
        "summarise",
        "overview",
        "brief",
        "explain the meeting",
        "what happened in this meeting"
    ]
    query = query.lower()
    if any(keyword in query for keyword in keywords):
        return full_transcript_pipeline(chunks)
    else :
        return normal_chunk_pipeline(query,collection)

In [20]:
query = "What he said about his customers"

In [21]:
answer = backend_pipeline(query,collection,chunks)

In [22]:
print(answer)

The speaker states that they preach to their clients every day that they don't care about the money, are not counting pennies, and could care less how much clients spend with them. They are there to develop a relationship with the client.
